###PHASE 7 — Recruiter Intelligence Scoring

Cross Encoder
+
Dense Similarity
+
BM25
+
Github Activity
+
Response Rate
+
Interview Completion
+
Open To Work
+
Notice Period
+
Saved By Recruiters

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pickle
import json
import numpy as np
import pandas as pd

In [3]:
DATA_PATH="/content/drive/MyDrive/Redrob_AI_Hackathon/datasets/candidates.jsonl"

candidates=[]

with open(DATA_PATH,'r',encoding='utf-8') as f:
    for line in f:
        candidates.append(json.loads(line))

In [4]:
candidate_dict = {}

for c in candidates:
    candidate_dict[c["candidate_id"]] = c

In [5]:
with open(
"/content/drive/MyDrive/Redrob_AI_Hackathon/outputs/top300_crossencoder.pkl",
"rb"
) as f:

    top300 = pickle.load(f)

In [6]:
cross_scores = np.array(
    [x["cross_score"] for x in top300]
)

cross_min = cross_scores.min()
cross_max = cross_scores.max()

for x in top300:

    x["cross_norm"] = (
        (x["cross_score"] - cross_min)
        /(cross_max-cross_min)
    )

In [7]:
def recruiter_score(candidate_entry):

    cid = candidate_entry["candidate_id"]

    cand = candidate_dict[cid]

    sig = cand["redrob_signals"]

    score = 0

    # Cross encoder (40%)
    score += 0.40 * candidate_entry["cross_norm"]

    # Github score (10%)
    github = max(sig["github_activity_score"],0)/100
    score += 0.10 * github

    # Recruiter response (15%)
    score += 0.15 * sig["recruiter_response_rate"]

    # Interview completion (10%)
    score += 0.10 * sig["interview_completion_rate"]

    # Saved by recruiters (10%)
    saved_score = min(
        sig["saved_by_recruiters_30d"]/20,
        1
    )

    score += 0.10 * saved_score

    # Open to work (5%)
    if sig["open_to_work_flag"]:
        score += 0.05

    # Notice period (10%)
    days = sig["notice_period_days"]

    if days <= 30:
        score += 0.10

    elif days <= 60:
        score += 0.07

    elif days <= 90:
        score += 0.04

    return score

In [8]:
for x in top300:

    x["final_score"] = recruiter_score(x)

In [9]:
top300 = sorted(
    top300,
    key=lambda x:x["final_score"],
    reverse=True
)

In [10]:
for i in range(10):

    print("="*80)

    print(top300[i]["candidate_id"])

    print("Final Score:",
          top300[i]["final_score"])

    print()

    print(top300[i]["text"][:1000])

CAND_0018499
Final Score: 0.8963000000000002

Current role: Senior Machine Learning Engineer.
Primary profession: Senior Machine Learning Engineer.
Current company: Zomato.
Industry: Food Delivery.
Location: Noida, Uttar Pradesh.
Professional experience of 7.2 years.
Professional Summary:
Senior AI engineer with 7.2 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector recall, serving 50M+ queries per month. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I've shipped systems in both early-stage product companies and at larger scale, and I've spent enough time on both that I know which tradeoffs apply where. I have strong opinions about when LLMs are the right hammer and w

In [11]:
PRODUCT_COMPANIES = {
    "Google",
    "Amazon",
    "Apple",
    "Flipkart",
    "CRED",
    "Zomato",
    "Paytm",
    "Razorpay",
    "Zoho",
    "Meesho",
    "Yellow.ai",
    "Freshworks",
    "InMobi",
    "Swiggy",
    "PhonePe",
    "Myntra"
}

In [12]:
def recruiter_score(candidate_entry):

    cid = candidate_entry["candidate_id"]

    cand = candidate_dict[cid]
    sig = cand["redrob_signals"]

    score = 0

    #########################################
    # 1. Cross Encoder (35%)
    #########################################

    score += 0.35 * candidate_entry["cross_norm"]

    #########################################
    # 2. Recruiter Response Rate (15%)
    #########################################

    score += 0.15 * sig["recruiter_response_rate"]

    #########################################
    # 3. Github Activity (10%)
    #########################################

    github_score = max(sig["github_activity_score"],0)

    score += 0.10 * min(github_score/100,1)

    #########################################
    # 4. Interview Completion (10%)
    #########################################

    score += 0.10 * sig["interview_completion_rate"]

    #########################################
    # 5. Saved by Recruiters (10%)
    #########################################

    score += 0.10 * min(
        sig["saved_by_recruiters_30d"]/20,
        1
    )

    #########################################
    # 6. Open To Work (5%)
    #########################################

    if sig["open_to_work_flag"]:
        score += 0.05

    #########################################
    # 7. Notice Period (5%)
    #########################################

    days = sig["notice_period_days"]

    if days <= 30:
        score += 0.05

    elif days <= 60:
        score += 0.04

    elif days <= 90:
        score += 0.02

    #########################################
    # 8. Experience Fit (5%)
    #########################################

    years = cand["profile"]["years_of_experience"]

    if 6 <= years <= 8:
        score += 0.05

    elif 5 <= years <= 9:
        score += 0.03

    #########################################
    # 9. Career Stability (3%)
    #########################################

    current_tenure = 0

    for job in cand["career_history"]:
        if job["is_current"]:
            current_tenure = job["duration_months"]
            break

    if 24 <= current_tenure <= 48:
        score += 0.03

    elif 12 <= current_tenure < 24:
        score += 0.015

    #########################################
    # 10. Product Company Bonus (2%)
    #########################################

    company = cand["profile"]["current_company"]

    if company in PRODUCT_COMPANIES:
        score += 0.02

    #########################################
    # Job hopping penalty
    #########################################

    num_jobs = len(cand["career_history"])

    if num_jobs >= 6:
        score -= 0.02

    return score

In [13]:
for x in top300:

    x["final_score"] = recruiter_score(x)

In [14]:
top300 = sorted(
    top300,
    key=lambda x:x["final_score"],
    reverse=True
)

In [15]:
for i in range(15):

    print("="*80)

    print(top300[i]["candidate_id"])

    print(
        round(top300[i]["final_score"],4)
    )

    print()

    print(top300[i]["text"][:800])

CAND_0018499
0.8963

Current role: Senior Machine Learning Engineer.
Primary profession: Senior Machine Learning Engineer.
Current company: Zomato.
Industry: Food Delivery.
Location: Noida, Uttar Pradesh.
Professional experience of 7.2 years.
Professional Summary:
Senior AI engineer with 7.2 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector recall, serving 50M+ queries per month. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I've shipped systems in both early-stage product companies and at l
CAND_0077337
0.8742

Current role: Staff Machine Learning Engineer.
Primary profession: Staff Machine Learning Engineer.
Current company: Paytm.
Industry: Fintech.
Location: Koch

In [16]:
submission=[]

rank=1

for x in top300:

    submission.append({

        "rank":rank,

        "candidate_id":x["candidate_id"]

    })

    rank+=1

submission_df=pd.DataFrame(submission)

In [17]:
submission_df.to_csv(
"/content/drive/MyDrive/Redrob_AI_Hackathon/outputs/submission.csv",
index=False
)